In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import window as W

spark = SparkSession.builder.appName("Pratice").getOrCreate()


**Q1. Load and inspect the data**

In [9]:
my_df = spark.read.csv("sales_orders.csv", header = True, inferSchema=True)
my_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- status: string (nullable = true)



In [8]:
my_df.show(10)

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|       2|      1800|           5|           UPI|Completed|
|    5005|       1004|2026-01-10|        Desk|  Furniture|       1|     12000|          15| Bank Transfer| Returned|
|    5006|       1005|2026-01-12|     Monitor|Electronics|      

In [ ]:
total_row = my_df.count()
print("Total row counts are: ", total_row)

Total row counts are:  30


Explicit schema can be preferable in a production pipeline
due to increase in performance and accuracy dtype

**Q2. Select, rename, and filte**

In [ ]:

df_q2 = my_df.withColumnRenamed("product", "product_name"). \
filter(
  (F.col("status") == "Completed") & (F.col("quantity") <=2)
).select(
    "order_id", "customer_id", "product_name", "quantity", "status"
)
df_q2.show()

+--------+-----------+------------+--------+---------+
|order_id|customer_id|product_name|quantity|   status|
+--------+-----------+------------+--------+---------+
|    5001|       1001|      Laptop|       1|Completed|
|    5002|       1002|  Headphones|       2|Completed|
|    5003|       1003|Office Chair|       1|Completed|
|    5004|       1001|    Keyboard|       2|Completed|
|    5006|       1005|     Monitor|       2|Completed|
|    5009|       1008|  Headphones|       1|Completed|
|    5010|       1009|     Monitor|       1|Completed|
|    5012|       1011|      Laptop|       1|Completed|
|    5014|       1002|    Keyboard|       1|Completed|
|    5015|       1003|        Desk|       1|Completed|
|    5017|       1005|      Laptop|       1|Completed|
|    5019|       1007|     Monitor|       2|Completed|
|    5020|       1008|Office Chair|       1|Completed|
|    5022|       1010|    Keyboard|       2|Completed|
|    5024|       1012|     Monitor|       1|Completed|
|    5025|

**Q3. Complex filter logic**

In [ ]:

df_d3 = my_df.filter(
    ((F.col("order_date") >= "2026-02-01") & (F.col("order_date") <= "2026-02-25")) &
     (((F.col("category") == "Electronics") & (F.col("discount_pct") >=5)) |
      (F.col("status") == "Pending"))
    )
df_d3.show()

+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+
|order_id|customer_id|order_date|   product|   category|quantity|unit_price|discount_pct|payment_method|   status|
+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+
|    5017|       1005|2026-02-07|    Laptop|Electronics|       1|     65000|           5|          Card|Completed|
|    5018|       1006|2026-02-09|Headphones|Electronics|       3|      2500|          10|           UPI| Returned|
|    5023|       1011|2026-02-22|      Desk|  Furniture|       1|     12000|          10|          Card|  Pending|
|    5024|       1012|2026-02-25|   Monitor|Electronics|       1|     14500|           5|           UPI|Completed|
+--------+-----------+----------+----------+-----------+--------+----------+------------+--------------+---------+



**Q4. Derived business metrics**

In [ ]:
df_q4 = my_df.withColumn("gross_amount", F.col("quantity") * F.col("unit_price")). \
withColumn("discount_amount",  (F.col("gross_amount") * F.col("discount_pct"))/100). \
withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))
df_q4_1 = df_q4.select("*")
df_q4_1.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|
|    5002|       1002|2026-01-04|  Headphones|Electronics|       2|      2500|           0|           UPI|Completed|        5000|            0.0|    5000.0|
|    5003|       1003|2026-01-05|Office Chair|  Furniture|       1|      8500|          10|          Card|Completed|        8500|          850.0|    7650.0|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|

**Q5. Conditional classification**

In [ ]:
my_df_q5 = df_q4.withColumn("order_value_band",
                            F.when(F.col("net_amount") > 20000, "High") \
                            .when(F.col("net_amount") >  5000, "Medium") \
                            .otherwise("Low")
                            ).select("net_amount", "order_value_band")
my_df_q5.show()


+----------+----------------+
|net_amount|order_value_band|
+----------+----------------+
|   61750.0|            High|
|    5000.0|             Low|
|    7650.0|          Medium|
|    3420.0|             Low|
|   10200.0|          Medium|
|   26100.0|            High|
|    1250.0|             Low|
|   11400.0|          Medium|
|    2500.0|             Low|
|   13775.0|          Medium|
|    2250.0|             Low|
|   58500.0|            High|
|   16150.0|          Medium|
|    1800.0|             Low|
|   10800.0|          Medium|
|    1000.0|             Low|
|   61750.0|            High|
|    6750.0|          Medium|
|   29000.0|            High|
|    8075.0|          Medium|
+----------+----------------+
only showing top 20 rows


 null net_amount values would be handled by lit(0)

**Q6. String and null handling**

In [ ]:
df_q6 = my_df.withColumn("product",
                         F.trim(F.lower("product"))
                         )
df_q6.show(200)

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+
|    5001|       1001|2026-01-03|      laptop|Electronics|       1|     65000|           5|          Card|Completed|
|    5002|       1002|2026-01-04|  headphones|Electronics|       2|      2500|           0|           UPI|Completed|
|    5003|       1003|2026-01-05|office chair|  Furniture|       1|      8500|          10|          Card|Completed|
|    5004|       1001|2026-01-08|    keyboard|Electronics|       2|      1800|           5|           UPI|Completed|
|    5005|       1004|2026-01-10|        desk|  Furniture|       1|     12000|          15| Bank Transfer| Returned|
|    5006|       1005|2026-01-12|     monitor|Electronics|      

 a safe approach to handling nulls in a selected string column without converting the literal string 'null' into a
valid business value - Using when contition check of null and updating the required value like., None, 0, Unknown

**Q7. Read options and schema validation**

In [ ]:
df_q7 = spark.read.csv("customers.csv", header = True, inferSchema=True)
df_q7.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- account_manager: string (nullable = true)



Handle malformed rows - if rows data is empty, row data has valid data with appropriate data type, invaild charaters.
validate before processing the file - where column data's are exist in the rows and its a vaild data or not

**Q8. Write partitioned output**

In [19]:
df_q8 = my_df.write.mode("overwrite").partitionBy("category").parquet("parquet_orders")
df_q8_show = spark.read.parquet("parquet_orders")
df_q8_show.show()

+--------+-----------+----------+------------+--------+----------+------------+--------------+---------+-----------+
|order_id|customer_id|order_date|     product|quantity|unit_price|discount_pct|payment_method|   status|   category|
+--------+-----------+----------+------------+--------+----------+------------+--------------+---------+-----------+
|    5001|       1001|2026-01-03|      Laptop|       1|     65000|           5|          Card|Completed|Electronics|
|    5002|       1002|2026-01-04|  Headphones|       2|      2500|           0|           UPI|Completed|Electronics|
|    5004|       1001|2026-01-08|    Keyboard|       2|      1800|           5|           UPI|Completed|Electronics|
|    5006|       1005|2026-01-12|     Monitor|       2|     14500|          10|          Card|Completed|Electronics|
|    5009|       1008|2026-01-17|  Headphones|       1|      2500|           0|          Card|Completed|Electronics|
|    5010|       1009|2026-01-19|     Monitor|       1|     1450

Using parquet will result in low memory storage and improve the spped process and very effective in costwise.
It will create a large number of small paritions files which lead to diffculties on maintainables.  

**Q9. Multi-level aggregation**

In [ ]:
my_df_aggregate = df_q4.groupBy( "order_id", "product", "category", "status").agg(
    F.count("order_id").alias("order_count"),
    F.sum("quantity").alias("total_quantity"),
    F.sum(
        F.col("quantity") * F.col("unit_price")
        ).alias("gross_revenue"),
    F.sum(
        (F.col("quantity") * F.col("unit_price")) - F.col("discount_amount")
        ).alias("net_revenue"),
    F.round(F.sum(
        (F.col("quantity") * F.col("unit_price")) - F.col("discount_amount"))
        / F.count("order_id"), 2
        ).alias("average_net_order_value")
    ).orderBy(F.col("net_revenue").desc())
    #(F.col("price") * F.col("quantity")).alias("gross_revenue"))
my_df_aggregate.select(
    "order_id", "product", "category", "order_count", "total_quantity", "gross_revenue",
    "net_revenue", "average_net_order_value", "status").show()

+--------+------------+-----------+-----------+--------------+-------------+-----------+-----------------------+---------+
|order_id|     product|   category|order_count|total_quantity|gross_revenue|net_revenue|average_net_order_value|   status|
+--------+------------+-----------+-----------+--------------+-------------+-----------+-----------------------+---------+
|    5025|      Laptop|Electronics|          1|             1|        65000|    65000.0|                65000.0|Completed|
|    5001|      Laptop|Electronics|          1|             1|        65000|    61750.0|                61750.0|Completed|
|    5017|      Laptop|Electronics|          1|             1|        65000|    61750.0|                61750.0|Completed|
|    5012|      Laptop|Electronics|          1|             1|        65000|    58500.0|                58500.0|Completed|
|    5019|     Monitor|Electronics|          1|             2|        29000|    29000.0|                29000.0|Completed|
|    5006|     M

**Q10. Conditional aggregation**

In [ ]:
my_df_q10 = my_df_aggregate.groupBy("status").agg(
    F.count("order_id").alias("total_order"),
    F.sum(
        F.when(
            F.col("status").isin("Completed"),
            F.col("net_revenue")
        ).when(
            F.col("status").isin("Pending"),
            F.col("net_revenue")
        ).when(
            F.col("status").isin("Returned"),
            F.col("net_revenue")
        ).otherwise("0")

    ).alias("total_net_revenue")


)
my_df_q10.show()

+---------+-----------+-----------------+
|   status|total_order|total_net_revenue|
+---------+-----------+-----------------+
| Returned|          3|          18570.0|
|Completed|         24|         405220.0|
|  Pending|          3|          38350.0|
+---------+-----------+-----------------+



Returned orders are 3 and it should not be included in business revenue since returned are not comes under business revenue.

**Q11. Customer performance**

In [ ]:
from ast import alias
my_df_join = df_q4_1.join(df_q7, on="customer_id", how="inner").filter(F.col("status") == "Completed"). \
groupBy("customer_id",
"customer_name", "city").agg(F.count("order_id").alias("Total_order"),
                             F.sum(F.col("net_amount")).alias("total_net_amount")).orderBy(
    F.col("total_net_amount").desc()
)

my_df_join.show(5)

+-----------+-------------+---------+-----------+----------------+
|customer_id|customer_name|     city|Total_order|total_net_amount|
+-----------+-------------+---------+-----------+----------------+
|       1001|        Aarav|  Chennai|          3|        130170.0|
|       1005|       Ishaan|Hyderabad|          3|         92600.0|
|       1011|       Vivaan|    Delhi|          1|         58500.0|
|       1007|        Rohan|  Chennai|          1|         29000.0|
|       1003|        Kabir|Bengaluru|          3|         19950.0|
+-----------+-------------+---------+-----------+----------------+
only showing top 5 rows


**Q12. Rank within a category**

In [ ]:
window_partition = W.Window.partitionBy("category").orderBy(F.col("net_amount").desc())
my_df_rank = df_q4_1.withColumn("rank_product", F.rank().over(window_partition))
my_df_rank.select("category", "product", "net_amount", "rank_product").show()

+-----------+------------+----------+------------+
|   category|     product|net_amount|rank_product|
+-----------+------------+----------+------------+
|Electronics|      Laptop|   65000.0|           1|
|Electronics|      Laptop|   61750.0|           2|
|Electronics|      Laptop|   61750.0|           2|
|Electronics|      Laptop|   58500.0|           4|
|Electronics|     Monitor|   29000.0|           5|
|Electronics|     Monitor|   26100.0|           6|
|Electronics|     Monitor|   13775.0|           7|
|Electronics|     Monitor|   13775.0|           7|
|Electronics|  Headphones|    6750.0|           9|
|Electronics|  Headphones|    5000.0|          10|
|Electronics|  Headphones|    4750.0|          11|
|Electronics|    Keyboard|    3600.0|          12|
|Electronics|    Keyboard|    3420.0|          13|
|Electronics|  Headphones|    2500.0|          14|
|Electronics|    Keyboard|    1800.0|          15|
|Electronics|    Keyboard|    1620.0|          16|
|  Furniture|Office Chair|   16

**rank      :** Rank will display the ranking number sequence in continues order. If there is same rank for particular rank number then same rank number will be assigned followed by skipping the next rank number.
Example Rank: 1, 2, 3, 3, 5.  

**dense_rank:** Dense rank will display the ranking number sequence in continues order. If there is same rank for particular rank number then same rank number will be assigned and it does not allow to skip the next rank number.
Example Rank: 1, 2, 3, 3, 4.

**row_number:** Row Number will display display the ranking number sequence in continues order which allow us to filter the records based on the row number

**Q13. Running total**

In [ ]:
w_p_running_total = W.Window.partitionBy("customer_id").orderBy("order_date", "order_id")
running_total_df = df_q4_1.withColumn("running_total_col", F.sum("net_amount").over(w_p_running_total))
running_total_df.show()
running_total_df.printSchema()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|running_total_col|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-----------------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|          61750.0|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|       2|      1800|           5|           UPI|Completed|        3600|          180.0|    3420.0|          65170.0|
|    5025|       1001|2026-03-02|      Laptop|Electronics|       1|     65000|           0|          Card|Completed|       65

**Q14. Previous order comparison**

In [ ]:

w_p_lag_by_date = W.Window.partitionBy("customer_id").orderBy("order_id")
lag_df = df_q4_1.withColumn(
                                "Previous_order_date",
                                F.lag("order_date", 1).over(w_p_lag_by_date)
                            ).withColumn(
                                "Previous_net_amount",
                                F.lag("net_amount", 1, "0").over(w_p_lag_by_date)
                            ).withColumn("diff_current_previous_net_amt",
                                (F.col("net_amount") - F.col("Previous_net_amount"))
                            )
lag_df.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-------------------+-------------------+-----------------------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|Previous_order_date|Previous_net_amount|diff_current_previous_net_amt|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+-------------------+-------------------+-----------------------------+
|    5001|       1001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|       65000|         3250.0|   61750.0|               NULL|                0.0|                      61750.0|
|    5004|       1001|2026-01-08|    Keyboard|Electronics|       2|      1800|           5| 

**Q15. Latest order per customer**

In [ ]:
w_p_latest_order = W.Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())
latest_order_df = df_q4_1.withColumn("latest_order",
                                  F.row_number().over(w_p_latest_order)
                                  ).filter(F.col("latest_order") == 1)
latest_order_df.show()

+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+------------+---------------+----------+------------+
|order_id|customer_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|gross_amount|discount_amount|net_amount|latest_order|
+--------+-----------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+------------+---------------+----------+------------+
|    5025|       1001|2026-03-02|      Laptop|Electronics|       1|     65000|           0|          Card|Completed|       65000|            0.0|   65000.0|       65000|            0.0|   65000.0|           1|
|    5026|       1002|2026-03-04|Office Chair|  Furniture|       1|      8500|           5|           UPI|Completed|        8500|          425.0|    8075.0|    

**Q16. Register and query a view**

In [ ]:
df_q4_1.createOrReplaceTempView("df_to_temp_view")
Temp_view_sql = spark.sql("""
select
MONTH(order_date) as month, sum(net_amount) as monthly_net_revenue
from df_to_temp_view
where status == "Completed"
group by MONTH(order_date)
order by month
""")
Temp_view_sql.show()

+-----+-------------------+
|month|monthly_net_revenue|
+-----+-------------------+
|    1|           182195.0|
|    2|           131700.0|
|    3|            91325.0|
+-----+-------------------+



**Q17. SQL versus DataFrame API**

In [ ]:
df_q4_1.createOrReplaceTempView("total_net_temp_view")
temp_view_total_net = spark.sql("""
    select payment_method,
    sum(net_amount) as total_net_revenue, status
    from total_net_temp_view
    where status == "Completed"
    group by payment_method, status
""")
temp_view_total_net.show()

+--------------+-----------------+---------+
|payment_method|total_net_revenue|   status|
+--------------+-----------------+---------+
|          Cash|           7900.0|Completed|
|           UPI|          49445.0|Completed|
|          Card|         327800.0|Completed|
| Bank Transfer|          20075.0|Completed|
+--------------+-----------------+---------+



In [ ]:

df_total_net_amt_by_payment = df_q4_1.groupBy("payment_method", "status").agg(
    F.sum(F.col("net_amount")).alias("total_net_revenue")
    ).filter(F.col("status") == "Completed").select("payment_method", "total_net_revenue","status")
df_total_net_amt_by_payment.show()

+--------------+-----------------+---------+
|payment_method|total_net_revenue|   status|
+--------------+-----------------+---------+
|          Cash|           7900.0|Completed|
|           UPI|          49445.0|Completed|
|          Card|         327800.0|Completed|
| Bank Transfer|          20075.0|Completed|
+--------------+-----------------+---------+



**Q18. Join types and unmatched keys**

In [ ]:
inner_join_df_q18 = my_df.join(df_q4, on="customer_id", how="inner")
inner_join_df_q18.show()

+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|customer_id|order_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|order_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|
+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|       1001|    5025|2026-03-02|      Laptop|Electronics|       1|     65000|           0|          Card|Completed|    5001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed| 

In [ ]:
left_join_df_q18 = my_df.join(df_q4, on="customer_id", how="left")
left_join_df_q18.show()

+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|customer_id|order_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|order_id|order_date|     product|   category|quantity|unit_price|discount_pct|payment_method|   status|gross_amount|discount_amount|net_amount|
+-----------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+--------+----------+------------+-----------+--------+----------+------------+--------------+---------+------------+---------------+----------+
|       1001|    5001|2026-01-03|      Laptop|Electronics|       1|     65000|           5|          Card|Completed|    5025|2026-03-02|      Laptop|Electronics|       1|     65000|           0|          Card|Completed| 

inner join will preserves all the matching customer_id rows from left and right join
left join will preserves all the matching and non matching customer_id rows

iNull() to identify which records each
preserves. Write a check to i

**Q19. Avoiding duplicate-column and row-count issues**

In [ ]:
left_join_df = my_df.alias("o")
right_join_df = df_q4.alias("c")

inner_join_df_q18 = left_join_df.join(right_join_df, on="customer_id", how="inner")
inner_join_df_q18.select("c.customer_id", "o.order_id", "c.product", "c.category", "c.quantity", "c.status").show()

+-----------+--------+------------+-----------+--------+---------+
|customer_id|order_id|     product|   category|quantity|   status|
+-----------+--------+------------+-----------+--------+---------+
|       1001|    5025|      Laptop|Electronics|       1|Completed|
|       1001|    5004|      Laptop|Electronics|       1|Completed|
|       1001|    5001|      Laptop|Electronics|       1|Completed|
|       1002|    5026|  Headphones|Electronics|       2|Completed|
|       1002|    5014|  Headphones|Electronics|       2|Completed|
|       1002|    5002|  Headphones|Electronics|       2|Completed|
|       1003|    5027|Office Chair|  Furniture|       1|Completed|
|       1003|    5015|Office Chair|  Furniture|       1|Completed|
|       1003|    5003|Office Chair|  Furniture|       1|Completed|
|       1001|    5025|    Keyboard|Electronics|       2|Completed|
|       1001|    5004|    Keyboard|Electronics|       2|Completed|
|       1001|    5001|    Keyboard|Electronics|       2|Comple

counting the customer and order rows cusotmer_id when any number of cusomter_id exist more in the records then that checks customer_id uniqueness before joining.